# Portfolio Allocation From Prediction Markets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crowdvector/polybridge-cookbooks/blob/main/portfolio-allocation/portfolio-allocation.ipynb)

Use PolyBridge Forecast probabilities and Yahoo Finance covariance data to generate a live portfolio allocation snapshot.

The Colab and GitHub links above are the final public paths. They will work after `crowdvector/polybridge-cookbooks` is published.

These results are market-implied snapshots, not financial advice.

In [ ]:
%pip install --quiet requests pandas numpy scipy matplotlib yfinance

In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def resolve_workdir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "portfolio-allocation",
        Path.cwd().parent,
    ]
    for candidate in candidates:
        helper = candidate / "portfolio_optimizer.py"
        if helper.exists():
            return helper.parent

    repo_root = Path("/content/polybridge-cookbooks")
    helper = repo_root / "portfolio-allocation" / "portfolio_optimizer.py"
    if helper.exists():
        return helper.parent

    try:
        subprocess.run(
            ["git", "clone", "https://github.com/crowdvector/polybridge-cookbooks.git", str(repo_root)],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
    except subprocess.CalledProcessError as exc:
        raise FileNotFoundError(
            "portfolio_optimizer.py is not present locally. The fallback clone path works after "
            "crowdvector/polybridge-cookbooks is published."
        ) from exc

    helper = repo_root / "portfolio-allocation" / "portfolio_optimizer.py"
    if not helper.exists():
        raise FileNotFoundError("portfolio_optimizer.py was not found after cloning the public repo.")
    return helper.parent


workdir = resolve_workdir()
os.chdir(workdir)
if str(workdir) not in sys.path:
    sys.path.insert(0, str(workdir))

import portfolio_optimizer

workdir

In [ ]:
paths, snapshot = portfolio_optimizer.run_once(prompt_for_key=True)
paths

In [ ]:
import pandas as pd

scenario_df = pd.DataFrame(
    {
        "probability": snapshot["scenario_probabilities"],
        "normalized_weight": snapshot["scenario_normalized_weights"],
        "source_market_count": snapshot["scenario_source_market_counts"],
    }
).sort_index()

allocation_df = pd.DataFrame(
    {
        "expected_return": snapshot["expected_returns_per_asset"],
        "equal_weight": snapshot["equal_weight_baseline"],
        "optimized_weight": snapshot["optimized_allocation"],
        "tilt_vs_equal": snapshot["tilt_vs_equal"],
    }
).sort_index()

scenario_df.style.format({
    "probability": "{:.2%}",
    "normalized_weight": "{:.2%}",
})

In [ ]:
allocation_df.style.format({
    "expected_return": "{:+.2%}",
    "equal_weight": "{:.2%}",
    "optimized_weight": "{:.2%}",
    "tilt_vs_equal": "{:+.2%}",
})

In [ ]:
from IPython.display import Image, display, Markdown

display(Markdown(f"Snapshot JSON: `{paths.snapshot}`"))
display(Markdown(f"Summary JSON: `{paths.summary}`"))
display(Markdown(f"Covariance CSV: `{paths.covariance_csv}`"))
display(Image(filename=str(paths.scenario_chart)))
display(Image(filename=str(paths.implied_returns_chart)))
display(Image(filename=str(paths.allocation_chart)))

Generated assets are written to `portfolio-allocation/assets/`:

- `snapshot.json`
- `allocation-summary.json`
- `covariance-matrix.csv`
- `scenario-probabilities.png`
- `implied-return-distributions.png`
- `allocation-output.png`

The snapshot, charts, and optimization output are live market-implied outputs and should not be treated as financial advice.